# 🧠 Neural Network Experiment Dashboard

Interactive control panel for training, visualizing, and comparing neural network architectures.

**Features:**
- Change hyperparameters live
- Select architecture (MLP, CNN, AnomalyMLP)
- Restart training dynamically
- Visualize training progress in real time

In [ ]:
import sys
import threading
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# nnviz imports
from nnviz.engine.models import ModelFactory
from nnviz.engine.trainer import Trainer, TrainerConfig
from nnviz.engine.hooks import HookManager
from nnviz.visualization.matplotlib_backend import MatplotlibBackend
from nnviz.analysis.weight_evolution import WeightTracker
from nnviz.ui.notebook_utils import (
    create_architecture_selector,
    create_dataset_selector,
    create_hyperparameter_panel,
    create_training_controls,
    create_live_plot_output,
    create_status_bar,
    create_progress_bar,
    update_status,
)

viz = MatplotlibBackend()
print('✅ All imports successful')

In [ ]:
# ── Dashboard Setup ──────────────────────────────────────────────────────
arch_sel = create_architecture_selector()
dataset_sel = create_dataset_selector()
hp = create_hyperparameter_panel()
plot_out = create_live_plot_output()
metrics_out = create_live_plot_output()
status = create_status_bar()
progress = create_progress_bar()

# State
training_state = {'running': False, 'model': None, 'history': {}, 'tracker': None}

def get_config():
    """Read current widget values into a config."""
    hidden = [int(x.strip()) for x in hp['hidden_sizes'].value.split(',') if x.strip()]
    return {
        'arch': arch_sel.value,
        'dataset': dataset_sel.value,
        'lr': hp['learning_rate'].value,
        'batch_size': hp['batch_size'].value,
        'epochs': hp['epochs'].value,
        'dropout': hp['dropout'].value,
        'hidden_sizes': hidden,
    }

In [ ]:
# ── Training Logic ────────────────────────────────────────────────────────
def train_model():
    cfg = get_config()
    update_status(status, f'Creating {cfg["arch"]} model...', '#ff0')
    
    # Create model
    model_kwargs = {}
    if cfg['arch'] in ('mlp', 'anomaly_mlp'):
        model_kwargs['hidden_sizes'] = cfg['hidden_sizes']
        model_kwargs['dropout'] = cfg['dropout']
    model = ModelFactory.create(cfg['arch'], **model_kwargs)
    training_state['model'] = model
    
    # Weight tracker
    tracker = WeightTracker(model)
    training_state['tracker'] = tracker
    
    # Create dummy data for demo
    from torch.utils.data import DataLoader, TensorDataset
    if cfg['arch'] in ('cnn',):
        X = torch.randn(256, 1, 28, 28)
    else:
        X = torch.randn(256, 784) if cfg['arch'] == 'mlp' else torch.randn(256, 41)
    y = torch.randint(0, 10 if cfg['arch'] != 'anomaly_mlp' else 2, (256,))
    loader = DataLoader(TensorDataset(X, y), batch_size=cfg['batch_size'], shuffle=True)
    
    # Train
    config = TrainerConfig(epochs=cfg['epochs'], learning_rate=cfg['lr'])
    trainer = Trainer(model, config=config)
    training_state['running'] = True
    
    history = {'train_loss': [], 'train_accuracy': []}
    
    for epoch in range(cfg['epochs']):
        if not training_state['running']:
            update_status(status, f'Stopped at epoch {epoch}', '#f00')
            break
        
        # Manual epoch
        model.train()
        epoch_loss, correct, total = 0.0, 0, 0
        for inputs, targets in loader:
            trainer.optimizer.zero_grad()
            outputs = model(inputs)
            loss = trainer.criterion(outputs, targets)
            loss.backward()
            trainer.optimizer.step()
            epoch_loss += loss.item()
            _, pred = outputs.max(1)
            correct += pred.eq(targets).sum().item()
            total += targets.size(0)
        
        avg_loss = epoch_loss / len(loader)
        accuracy = correct / total
        history['train_loss'].append(avg_loss)
        history['train_accuracy'].append(accuracy)
        tracker.snapshot(epoch)
        
        # Update progress
        progress.value = (epoch + 1) / cfg['epochs'] * 100
        update_status(status, f'Epoch {epoch+1}/{cfg["epochs"]} — Loss: {avg_loss:.4f} — Acc: {accuracy:.2%}', '#0f0')
        
        # Live plot
        with plot_out:
            clear_output(wait=True)
            fig = viz.plot_training_curves(history)
            plt.show()
    
    training_state['history'] = history
    training_state['running'] = False
    
    # Final visualizations
    with metrics_out:
        clear_output(wait=True)
        fig = viz.plot_weight_distribution(model)
        plt.show()
        fig = tracker.plot_evolution()
        plt.show()
    
    update_status(status, '✅ Training complete!', '#0ff')

def stop_training():
    training_state['running'] = False

def reset_training():
    training_state['running'] = False
    training_state['model'] = None
    training_state['history'] = {}
    progress.value = 0
    with plot_out:
        clear_output()
    with metrics_out:
        clear_output()
    update_status(status, 'Ready — Reset complete.', '#0ff')

In [ ]:
# ── Build Dashboard ──────────────────────────────────────────────────────
controls = create_training_controls(
    on_train=lambda: threading.Thread(target=train_model, daemon=True).start(),
    on_stop=stop_training,
    on_reset=reset_training,
)

header = widgets.HTML(
    '<h2 style="color:#0ff; font-family:monospace; margin:0 0 10px 0;">'
    '🧠 Neural Network Experiment Dashboard</h2>'
)

selector_row = widgets.HBox([arch_sel, dataset_sel])
hp_box = widgets.VBox(list(hp.values()))

tabs = widgets.Tab(children=[plot_out, metrics_out])
tabs.set_title(0, '📈 Training Curves')
tabs.set_title(1, '🔍 Analysis')

dashboard = widgets.VBox([
    header,
    selector_row,
    hp_box,
    controls,
    progress,
    status,
    tabs,
])

display(dashboard)

## How to Use

1. **Select** a model architecture from the dropdown
2. **Adjust** hyperparameters using the sliders
3. Click **▶ Train** to start training
4. Watch the **Training Curves** tab for live loss/accuracy updates
5. Click **⏹ Stop** to interrupt training at any time
6. Switch to the **Analysis** tab for weight distributions and evolution plots
7. Click **↻ Reset** to clear state and start a new experiment